# 16 (DE) — Data Quality Checks

**Data Engineer perspective.** The four audits every pipeline needs: completeness (NULLs), uniqueness (duplicates), referential integrity (orphans), and domain validity — plus quarantine of bad rows for review.

In [ ]:
import os
from dotenv import load_dotenv
from irispark import IrisParkSession

load_dotenv()

# Connection via environment variables (matches examples/basic_usage.py).
# Set IRIS_HOST / IRIS_PORT / IRIS_NAMESPACE / IRIS_USERNAME / IRIS_PASSWORD.
try:
    session = IrisParkSession.builder() \
        .host(os.environ.get("IRIS_HOST", "localhost")) \
        .port(int(os.environ.get("IRIS_PORT", 1972))) \
        .namespace(os.environ.get("IRIS_NAMESPACE", "USER")) \
        .username(os.environ.get("IRIS_USERNAME", "_SYSTEM")) \
        .password(os.environ.get("IRIS_PASSWORD", "SYS")) \
        .getOrCreate()
    print("Connected to IRIS:", session)
except Exception as e:
    print("SKIP: IRIS not reachable -", e)
    session = None

In [ ]:
if session is None:
    raise SystemExit("IRIS not reachable; skipping this notebook.")

## 1. Deliberately dirty data

Orders referencing missing customers, duplicated keys, NULL amounts, and an invalid region.

In [ ]:
import pandas as pd

clientes = session.createDataFrame(pd.DataFrame({
    "cliente_id": [1, 2, 3, 4, 5],
    "nome": ["Ana", "Bruno", "Carlos", "Diana", "Eva"],
}))

pedidos = session.createDataFrame(pd.DataFrame({
    "pedido_id": [1, 2, 2, 3, 4, 5, 6],
    "cliente_id": [1, 2, 2, 99, None, 3, 4],   # 99 = orphan; None = missing FK
    "regiao": ["SP", "RJ", "RJ", "SP", "XX", "MG", "RS"],  # XX invalid
    "valor": [100.0, None, None, 250.0, 300.0, None, 90.0],
}))
print("pedidos:", pedidos.count(), "| clientes:", clientes.count())

## 2. Completeness audit

NULL counts per critical column, computed as one aggregate.

In [ ]:
from irispark.functions import col, count, when

pedidos.agg(
    count(when(col("pedido_id").isNull(), 1)).alias("pedido_id_nulls"),
    count(when(col("cliente_id").isNull(), 1)).alias("cliente_id_nulls"),
    count(when(col("valor").isNull(), 1)).alias("valor_nulls"),
).show()

## 3. Uniqueness audit

Duplicate business keys via grouped aggregate with a HAVING-routed filter.

In [ ]:
from irispark.functions import count as fcount, lit

dupes = pedidos.groupBy("pedido_id").agg(fcount(lit(1)).alias("n")).filter("n > 1")
dupes.show()

n_all = pedidos.count()
n_distinct = pedidos.dropDuplicates(["pedido_id"]).count()
print(f"duplicate rows to resolve: {n_all - n_distinct}")

## 4. Referential integrity — orphan detection with `left_anti`

Orders whose customer does not exist. `left_anti` returns exactly the left rows with no match.

In [ ]:
orphan_items = pedidos.join(clientes, "cliente_id", "left_anti") \
                      .filter("cliente_id IS NOT NULL")
orphan_items.select("pedido_id", "cliente_id").show()
print("orphans:", orphan_items.count())

## 5. Domain validity

`isin` whitelists; negation catches the violations.

In [ ]:
REGIOES_VALIDAS = ["SP", "RJ", "MG", "RS"]
invalidos = pedidos.filter(~col("regiao").isin(REGIOES_VALIDAS))
invalidos.select("pedido_id", "regiao").show()

## 6. Quarantine pattern

Bad rows go to a review table; clean rows flow downstream. Arms are column-normalized before the union, and the quarantine set is materialized so the clean-flow join reads a physical table.

In [ ]:
from irispark.functions import col, count as fcount, lit

REGIOES_VALIDAS = ["SP", "RJ", "MG", "RS"]
cols = ["pedido_id", "cliente_id", "regiao", "valor"]

orfaos = pedidos.join(clientes, "cliente_id", "left_anti").select(*cols)
invalidos = pedidos.filter(~col("regiao").isin(REGIOES_VALIDAS)).select(*cols)
nulos = pedidos.filter(col("valor").isNull()).select(*cols)

quarentena = orfaos.union(invalidos).union(nulos).dropDuplicates(["pedido_id"])
quarentena.write.mode("overwrite").saveAsTable("dq_quarentena_demo")

quarentena_t = session.table("dq_quarentena_demo")
limpos = pedidos.join(quarentena_t.select("pedido_id"), "pedido_id", "left_anti")
print("quarantined:", quarentena_t.count(), "| clean flow:", limpos.count())

## 7. Reconciliation

The invariant a pipeline must prove: quarantined + clean == source.

In [ ]:
total = pedidos.dropDuplicates(["pedido_id"]).count()
q = session.table("dq_quarentena_demo").count()
c = pedidos.join(session.table("dq_quarentena_demo").select("pedido_id"), "pedido_id", "left_anti").count()
print(f"{q} + {c} == {total}: {q + c == total}")

## 8. Cleanup

In [ ]:
session.sql("DROP TABLE IF EXISTS dq_quarentena_demo")
print("dropped dq_quarentena_demo")

In [ ]:
if session is not None:
    session.close()
    print("Session closed.")